# অগ্রাধিকার সদস্য মধ্যবর্তী সফ্টওয়্যার দিয়ে হোটেল বুকিং

এই নোটবুকটি মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক ব্যবহার করে **ফাংশন-ভিত্তিক মধ্যবর্তী সফ্টওয়্যার** প্রদর্শন করে। আমরা শর্তসাপেক্ষ ওয়ার্কফ্লো উদাহরণের উপর ভিত্তি করে একটি মধ্যবর্তী স্তর যোগ করি যা অগ্রাধিকার সদস্যদের বিশেষ সুবিধা দেয়।

## আপনি যা শিখবেন:
1. **ফাংশন-ভিত্তিক মধ্যবর্তী সফ্টওয়্যার**: ফাংশনের ফলাফল ধারণ এবং পরিবর্তন করা
2. **পরিপ্রেক্ষিত অ্যাক্সেস**: কার্যক্রমের পরে `context.result` পড়া এবং পরিবর্তন করা
3. **বিজনেস লজিক বাস্তবায়ন**: অগ্রাধিকার সদস্য সুবিধা
4. **ফলাফল ওভাররাইড**: ব্যবহারকারীর অবস্থা অনুযায়ী ফাংশনের ফলাফল পরিবর্তন
5. **একই ওয়ার্কফ্লো, ভিন্ন ফলাফল**: মধ্যবর্তী সফ্টওয়্যার দ্বারা পরিচালিত আচরণ পরিবর্তন

## মধ্যবর্তী সফ্টওয়্যার সহ ওয়ার্কফ্লো স্থাপত্য:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## শর্তসাপেক্ষ ওয়ার্কফ্লো থেকে মূল পার্থক্য:

**মধ্যবর্তী সফ্টওয়্যার ছাড়া** (14-conditional-workflow.ipynb):
- প্যারিসে কোন রুম নেই → বিকল্প এজেন্টের কাছে রুট

**মধ্যবর্তী সফ্টওয়্যার সহ** (এই নোটবুক):
- নিয়মিত ব্যবহারকারী + প্যারিস → কোন রুম নেই → বিকল্প এজেন্টের কাছে রুট
- অগ্রাধিকার ব্যবহারকারী + প্যারিস → 🌟 মধ্যবর্তী সফ্টওয়্যার ওভাররাইড করে! → উপলব্ধ → বুকিং এজেন্টের কাছে রুট

## পূর্বশর্ত:
- মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক ইনস্টল করা
- শর্তসাপেক্ষ ওয়ার্কফ্লো এর ধারণা (দেখুন 14-conditional-workflow.ipynb)
- GitHub টোকেন অথবা OpenAI API কী
- মধ্যবর্তী সফ্টওয়্যার প্যাটার্নের মৌলিক ধারণা


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ধাপ ১: কাঠামোবদ্ধ আউটপুটের জন্য পিড্যান্টিক মডেল সংজ্ঞায়িত করুন

এই মডেলগুলি সেই **স্কিমা** সংজ্ঞায়িত করে যা এজেন্টরা ফেরত দেবে। আমরা একটি `priority_override` ফিল্ড যোগ করেছি যা ট্র্যাক করবে কখন মিডলওয়্যার উপলব্ধতা ফলাফল পরিবর্তন করে।


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ধাপ ২: অগ্রাধিকার সদস্যের ডাটাবেস সংজ্ঞায়িত করুন

এই ডেমোর জন্য, আমরা একটি অগ্রাধিকার সদস্যপদ ডাটাবেস সিমুলেট করব। প্রোডাকশনে, এটি একটি বাস্তব ডাটাবেস বা API থেকে তথ্য নেবে।

**অগ্রাধিকার সদস্যরা:**
- `alice@example.com` - ভিআইপি সদস্য
- `bob@example.com` - প্রিমিয়াম সদস্য  
- `priority_user` - টেস্ট অ্যাকাউন্ট


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## ধাপ ৩: হোটেল বুকিং টুল তৈরি করুন

শর্তাধীন কর্মপ্রবাহের মতোই, তবে এখন এটি মিডলওয়্যার দ্বারা আটকানো হবে!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ধাপ ৪: 🌟 প্রায়োরিটি চেক মিডলওয়্যার তৈরি করুন (মুখ্য বৈশিষ্ট্য!)

এটি এই নোটবুকের **মূখ্য কার্যকারিতা**। মিডলওয়্যারটি:

১. **ইন্টারসেপ্ট** করে hotel_booking ফাংশন কলটি
২. `next(context)` কল করে ফাংশনটি স্বাভাবিকভাবে **নির্বাহ করে**
৩. `context.result` এ ফলাফলটি **পরীক্ষা করে**
৪. যদি ব্যবহারকারী প্রায়োরিটি হয় এবং কোন রুম উপলব্ধ না থাকে, তাহলে ফলাফলটি **অতিরিক্ত লেখে**
৫. সংশোধিত ফলাফলটি এজেন্টের কাছে **ফিরিয়ে দেয়**

**মুখ্য প্যাটার্ন:**
```python
async def my_middleware(context, next):
    await next(context)  # ফাংশন সম্পাদন করুন
    # এখন context.result এ ফাংশনের আউটপুট রয়েছে
    if some_condition:
        context.result = new_value  # ওভাররাইড!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## ধাপ ৫: রাউটিংয়ের জন্য শর্ত ফাংশন সংজ্ঞায়িত করুন

একই শর্ত ফাংশন যেমন শর্তাধীন কর্মপ্রবাহে থাকে - তারা রাউটিং নির্ধারণ করার জন্য কাঠামোবদ্ধ আউটপুট পরীক্ষা করে।


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## ধাপ ৬: কাস্টম ডিসপ্লে এক্সিকিউটর তৈরি করুন

আগের মতো একই এক্সিকিউটর - চূড়ান্ত ওয়ার্কফ্লো আউটপুট প্রদর্শন করে।


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## পদক্ষেপ ৭: পরিবেশ ভেরিয়েবলগুলি লোড করুন

LLM ক্লায়েন্ট (Microsoft Foundry / Azure OpenAI) কনফিগার করুন।


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## ধাপ ৮: মിഡলওয়্যার সহ AI এজেন্ট তৈরি করুন

**মূল পার্থক্য:** যখন availability_agent তৈরি করি, তখন আমরা `middleware` প্যারামিটারটি দিয়ে থাকি!

এভাবেই আমরা priority_check_middleware এজেন্টের ফাংশন কল পাইলাইনে ইনজেক্ট করি।


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ধাপ ৯: ওয়ার্কফ্লো তৈরি করুন

আগের মতো একই ওয়ার্কফ্লো কাঠামো - উপলব্ধতার উপর নির্ভর করে শর্তাধীন রাউটিং।


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## ধাপ ১০: টেস্ট কেস ১ - প্যারিসের সাধারণ ব্যবহারকারী (কোনও ওভাররাইড ছাড়া)

একজন সাধারণ ব্যবহারকারী প্যারিস বুক করার চেষ্টা করেন → কোন রুম নেই → alternative_agent -এ রুট হয়


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## ধাপ ১১: টেস্ট কেস ২ - 🌟 প্রাধান্যপ্রাপ্ত ব্যবহারকারী প্যারিসে (ওভাররাইড সহ!)

একজন প্রাধান্যপ্রাপ্ত সদস্য প্যারিস বুক করার চেষ্টা করেন → প্রথমে কোন কক্ষ নেই → 🌟 মিডলওয়্যার ওভাররাইড করে! → booking_agent-এ রুটিং হয়

**এটাই মিডলওয়্যার ক্ষমতার মূল প্রদর্শনী!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## ধাপ ১২: টেস্ট কেস ৩ - স্টকহোমে অগ্রাধিকার ব্যবহারকারী (ইতিমধ্যে উপলব্ধ)

অগ্রাধিকার ব্যবহারকারী স্টকহোম চেষ্টা করে → রুম উপলব্ধ → ওভাররাইডের প্রয়োজন নেই → booking_agent এ রুট করে

এটি দেখায় যে মিডলওয়্যার শুধুমাত্র প্রয়োজনে কাজ করে!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## মূল বিষয়গুলি এবং মিডলওয়্যার ধারণা

### ✅ আপনি কী শিখেছেন:

#### **1. ফাংশন ভিত্তিক মিডলওয়্যার প্যাটার্ন**

মিডলওয়্যার একটি সাধারণ async ফাংশন ব্যবহার করে ফাংশন কলগুলি ইন্টারসেপ্ট করে:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ফাংশন 실행ের আগে
    print("Intercepting...")
    
    # ফাংশনটি চালান
    await next(context)
    
    # ফাংশন execution এর পরে - ফলাফল পরিদর্শন করুন
    if context.result:
        # প্রয়োজনে ফলাফল সংশোধন করুন
        context.result = modified_value
```

#### **2. কনটেক্সট অ্যাক্সেস এবং ফলাফল ওভাররাইড**

- `context.function` - কল করা ফাংশনটি অ্যাক্সেস করুন
- `context.arguments` - ফাংশনের আর্গুমেন্ট পড়ুন
- `context.kwargs` - অতিরিক্ত প্যারামিটার অ্যাক্সেস করুন
- `await next(context)` - ফাংশনটি কার্যকর করুন
- `context.result` - ফাংশনের আউটপুট পড়ুন/পরিবর্তন করুন

#### **3. ব্যবসায়িক লজিক বাস্তবায়ন**

আমাদের মিডলওয়্যার প্রায়োরিটি সদস্য সুবিধা বাস্তবায়ন করে:
- **সাধারণ ব্যবহারকারীরা**: কোন পরিবর্তন নয়, স্ট্যান্ডার্ড ওয়ার্কফ্লো
- **প্রায়োরিটি ব্যবহারকারীরা**: "কোন উপলব্ধতা নেই" → "উপলব্ধ" ওভাররাইড
- **শর্তাধীন লজিক**: প্রয়োজন হলে মাত্র ওভাররাইড করে

#### **4. একই ওয়ার্কফ্লো, ভিন্ন ফলাফল**

মিডলওয়্যারের শক্তি:
- ✅ ওয়ার্কফ্লো কাঠামোতে কোন পরিবর্তন নেই
- ✅ টুল ফাংশনে কোন পরিবর্তন নেই
- ✅ শর্তাধীন রাউটিং লজিকে কোন পরিবর্তন নেই
- ✅ কেবল মিডলওয়্যার → ভিন্ন আচরণ!

### 🚀 বাস্তব জীবনের ব্যবহার:

১. **ভিআইপি/প্রিমিয়াম বৈশিষ্ট্যসমূহ**
   - প্রিমিয়াম ব্যবহারকারীদের জন্য রেট লিমিট ওভাররাইড করুন
   - সম্পদে প্রাধান্য অ্যাক্সেস প্রদান করুন
   - ডাইনামিক প্রিমিয়াম ফিচার আনলক করুন

২. **এ/বি পরীক্ষা**
   - ব্যবহারকারীদের ভিন্ন বাস্তবায়নে রুট করুন
   - নির্দিষ্ট ব্যবহারকারীদের জন্য নতুন ফিচার পরীক্ষা করুন
   - ধাপে ধাপে ফিচার রোলআউট

৩. **নিরাপত্তা ও সম্মতি**
   - ফাংশন কলগুলোর অডিট করুন
   - সংবেদনশীল অপারেশন ব্লক করুন
   - ব্যবসায়িক নিয়ম প্রয়োগ করুন

৪. **পারফরম্যান্স অপটিমাইজেশন**
   - নির্দিষ্ট ব্যবহারকারীর জন্য ফলাফল ক্যাশ করুন
   - সম্ভব হলে ব্যয়বহুল অপারেশন বাদ দিন
   - ডায়নামিক রিসোর্স বরাদ্দ

৫. **ত্রুটি হ্যান্ডলিং ও পুনরায় চেষ্টা**
   - ত্রুটি দমনে এবং সুচারুভাবে পরিচালনা করুন
   - পুনরায় চেষ্টা লজিক বাস্তবায়ন করুন
   - বিকল্প বাস্তবায়নে ফallback করুন

৬. **লগিং ও মনিটরিং**
   - ফাংশন কার্যকর করার সময় ট্র্যাক করুন
   - প্যারামিটার ও ফলাফল লগ করুন
   - ব্যবহার প্যাটার্ন মনিটর করুন

### 🔑 ডেকোরেটরের থেকে মূল পার্থক্যসমূহ:

| বৈশিষ্ট্য | ডেকোরেটর | middleware |
|---------|-----------|------------|
| **স্কোপ** | একটি ফাংশন | এজেন্টের সব ফাংশন |
| **নমনীয়তা** | সংজ্ঞায় নির্দিষ্ট | রানটাইমে ডায়নামিক |
| **কনটেক্সট** | সীমিত | পূর্ণ এজেন্ট কনটেক্সট |
| **সংযোজন** | একাধিক ডেকোরেটর | মিডলওয়্যার পাইপলাইন |
| **এজেন্ট-সচেতন** | না | হ্যাঁ (এজেন্ট স্টেট অ্যাক্সেস) |

### 📚 কখন মিডলওয়্যার ব্যবহার করবেন:

✅ **মিডলওয়্যার ব্যবহার করুন যখন:**
- ব্যবহারকারী/সেশন স্টেটের উপর ভিত্তি করে আচরণ পরিবর্তন করতে হবে
- একাধিক ফাংশনে লজিক প্রয়োগ করতে চান
- এজেন্ট স্তরের কনটেক্সটে অ্যাক্সেস প্রয়োজন
- আপনি ক্রস-কাটিং বিষয়াবলী (লগিং, অথ, ইত্যাদি) বাস্তবায়ন করছেন

❌ **মিডলওয়্যার ব্যবহার করবেন না যখন:**
- সাধারণ ইনপুট ভ্যালিডেশন (Pydantic ব্যবহার করুন)
- ফাংশন-নির্দিষ্ট লজিক (ফাংশনে রাখুন)
- একবারের পরিবর্তন (ফাংশন সরাসরি পরিবর্তন করুন)

### 🎓 উন্নত প্যাটার্নসমূহ:

```python
# একাধিক মিডলওয়্যার (কার্যকরী ক্রম গুরুত্বপূর্ণ!)
middleware=[
    logging_middleware,      # প্রথমে লগ হয়
    auth_middleware,         # তারপর অথেন্টিকেশন পরীক্ষা করে
    cache_middleware,        # তারপর ক্যাশ পরীক্ষা করে
    rate_limit_middleware,   # তারপর রেট সীমা প্রয়োগ করে
    priority_check_middleware  # অবশেষে অগ্রাধিকার পরীক্ষা
]

# শর্তসাপেক্ষ মিডলওয়্যার কার্যকরী
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ফলাফল পরিবর্তন করুন
    else:
        # সম্পূর্ণরূপে কার্যকরী এড়িয়ে যান
        context.result = cached_value
```

### 🔗 সম্পর্কিত ধারণাসমূহ:

- **এজেন্ট মিডলওয়্যার**: agent.run() কল ইন্টারসেপ্ট করে
- **ফাংশন মিডলওয়্যার**: টুল ফাংশন কলগুলি ইন্টারসেপ্ট করে (আমরা যেটা ব্যবহার করেছি!)
- **মিডলওয়্যার পাইপলাইন**: মধ্যবর্তী মিডলওয়্যার ধারাবাহিকভাবে কার্যকর হয়
- **কনটেক্সট প্রোপাগেশন**: মিডলওয়্যার চেইনের মাধ্যমে স্টেট পাস করা


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
